# Analisis Temporal de Precios - TFG

Este notebook usa los datos procesados para analizar patrones de dynamic pricing en portatiles de Amazon, PcComponentes y El Corte Inglés.

Bloques principales:
1. Evolucion diaria de precios por plataforma.
2. Ranking de productos con mayor variacion de precio.
3. Resumen ejecutivo para memoria del TFG.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

PROCESSED_DIR = BASE_DIR / "data" / "processed"
candidatos = sorted(PROCESSED_DIR.glob("precios_portatiles_procesado_*.csv"))
if not candidatos:
    raise FileNotFoundError("No hay CSV procesados en data/processed")

latest_csv = candidatos[-1]
print(f"CSV procesado cargado: {latest_csv}")

CSV procesado cargado: /Users/david/Documents/scraper/data/processed/precios_portatiles_procesado_20260330_1303.csv


In [ ]:
df = pd.read_csv(latest_csv)
df["fecha_extraccion"] = pd.to_datetime(df["fecha_extraccion"], errors="coerce")
df["fecha_dia"] = df["fecha_extraccion"].dt.date

# Filtro de limpieza: eliminar productos que no sean portátiles
palabras_excluidas = ['monitor', 'tablet', 'impresora', 'all in one', 'ipad', 'tab']
patron = '|'.join(palabras_excluidas)
df = df[~df['nombre'].str.lower().str.contains(patron, na=False)]

filas_iniciales = len(df)
print("Filas:", filas_iniciales)
print("Plataformas:", sorted(df["plataforma"].dropna().unique().tolist()))
df.head()

In [ ]:
evolucion = (
    df.groupby(["fecha_dia", "plataforma"], as_index=False)
      .agg(
          precio_medio=("precio_actual_num", "mean"),
          precio_mediano=("precio_actual_num", "median"),
          n_productos=("nombre", "count")
      )
)

display(evolucion.sort_values(["fecha_dia", "plataforma"]))

fig = px.line(
    evolucion,
    x="fecha_dia",
    y="precio_medio",
    color="plataforma",
    markers=True,
    title="Evolución diaria del precio medio por plataforma (Amazon, PcComponentes, El Corte Inglés)"
)
fig.update_layout(xaxis_title="Fecha", yaxis_title="Precio medio (EUR)")
fig.show()

In [ ]:
# Firma de producto para agrupar el mismo item a lo largo del tiempo
df["producto_id"] = (
    df["nombre"].astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str[:90]
)

variacion = (
    df.groupby(["plataforma", "producto_id"], as_index=False)
      .agg(
          observaciones=("precio_actual_num", "count"),
          precio_min=("precio_actual_num", "min"),
          precio_max=("precio_actual_num", "max")
      )
)

variacion["variacion_abs"] = (variacion["precio_max"] - variacion["precio_min"]).round(2)
variacion["variacion_pct"] = ((variacion["variacion_abs"] / variacion["precio_min"]) * 100).round(2)

ranking = variacion.sort_values("variacion_abs", ascending=False).head(20)
display(ranking)

fig_rank = px.bar(
    ranking.head(10),
    x="variacion_abs",
    y="producto_id",
    color="plataforma",
    orientation="h",
    title="Top 10 productos con mayor variación absoluta (Amazon, PcComponentes, El Corte Inglés)"
)
fig_rank.update_layout(xaxis_title="Variacion absoluta (EUR)", yaxis_title="Producto (id abreviado)")
fig_rank.show()

In [ ]:
resumen = (
    df.groupby("plataforma", as_index=False)
      .agg(
          productos=("nombre", "count"),
          precio_medio=("precio_actual_num", "mean"),
          precio_mediano=("precio_actual_num", "median"),
          precio_min=("precio_actual_num", "min"),
          precio_max=("precio_actual_num", "max"),
          descuento_medio_pct=("descuento_pct", "mean")
      )
)

resumen = resumen.round(2).sort_values("precio_medio")
display(resumen)

print("Resumen ejecutivo automatico (Amazon, PcComponentes, El Corte Inglés):")
print(f"- Muestra total analizada: {len(df)} observaciones")
for _, row in resumen.iterrows():
    print(
        f"- {row['plataforma']}: n={int(row['productos'])}, precio medio={row['precio_medio']} EUR, "
        f"rango=[{row['precio_min']}, {row['precio_max']}] EUR"
    )

if (variacion['variacion_abs'] > 0).any():
    top = variacion.sort_values('variacion_abs', ascending=False).iloc[0]
    print(
        f"- Mayor variacion detectada: {top['variacion_abs']} EUR en {top['plataforma']} (producto_id abreviado)."
    )
else:
    print("- Aun no hay variacion temporal significativa: se necesita acumular mas dias de extraccion.")